In [2]:
class Card:
    """
    Represents a single UNO card.
    color: 'Red', 'Blue', 'Green', 'Yellow'
    value: 0-9 or 'Skip'
    """
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def is_skip(self):
        return self.value == 'Skip'

    def matches(self, top_card):
        return self.color == top_card.color or self.value == top_card.value

c1 = Card('Red', 5)
c2 = Card('Blue', 5)
c3 = Card('Green', 3)
print(c1, 'matches', c2, '->', c1.matches(c2))
print(c1, 'matches', c3, '->', c1.matches(c3))

Red 5 matches Blue 5 -> True
Red 5 matches Green 3 -> False


In [3]:
import random
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    for color in colors:
        for number in range(10):
            deck.append(Card(color, number))
    for color in colors:
        deck.append(Card(color, 'Skip'))
    random.shuffle(deck)
    return deck

def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

def create_initial_state():
    deck = generate_deck()
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())
    top_card = deck.pop()
    while top_card.is_skip():
        deck.insert(0, top_card)
        top_card = deck.pop()
    return {'hands': hands, 'top_card': top_card, 'deck': deck, 'current': 0, 'skip_next': False}

In [4]:
def apply_move(state, player_idx, card):
    new_state = {
        'hands': [list(h) for h in state['hands']],
        'top_card': state['top_card'],
        'deck': list(state['deck']),
        'current': state['current'],
        'skip_next': False,
        'winner': state.get('winner', None)
    }
    if card is None:
        if new_state['deck']:
            new_state['hands'][player_idx].append(new_state['deck'].pop())
    else:
        new_state['hands'][player_idx].remove(card)
        new_state['top_card'] = card
        if card.is_skip():
            new_state['skip_next'] = True
        if len(new_state['hands'][player_idx]) == 0:
            new_state['winner'] = player_idx

    next_player = (player_idx + 1) % 3
    if new_state['skip_next']:
        next_player = (player_idx + 2) % 3
        new_state['skip_next'] = False
    new_state['current'] = next_player
    return new_state

random.seed(42)
state = create_initial_state()
print(f"Before - P1 hand: {state['hands'][0]}")
valid = get_valid_moves(state['hands'][0], state['top_card'])
print(f"Valid moves: {valid}")
if valid:
    state2 = apply_move(state, 0, valid[0])
    print(f"After playing {valid[0]} - P1 hand: {state2['hands'][0]}")
    print(f"New top card: {state2['top_card']}")

Before - P1 hand: [Red Skip, Blue 7, Red 8, Red 5, Blue Skip]
Valid moves: [Blue 7, Blue Skip]
After playing Blue 7 - P1 hand: [Red Skip, Red 8, Red 5, Blue Skip]
New top card: Blue 7


In [6]:
def evaluate(state, player_idx, strategy='defensive'):

    hand = state['hands'][player_idx]
    opp_indices = [i for i in range(3) if i != player_idx]
    c_opp = sum(len(state['hands'][i]) for i in opp_indices) / 2
    c_ai = len(hand)
    skips = sum(1 for c in hand if c.is_skip())

    if strategy == 'defensive':
        return 50 - 4*c_ai + 1*c_opp + 5*skips
    else:
        return 50 - 7*c_ai + 3*c_opp + 2*skips

random.seed(42)
state = create_initial_state()
print(f"P1 defensive score: {evaluate(state, 0, 'defensive')}")
print(f"P2 offensive score: {evaluate(state, 1, 'offensive')}")

P1 defensive score: 45.0
P2 offensive score: 30.0


In [7]:
minimax_tree_log = []

def minimax(state, depth, player_idx, is_maximizing, ai_player, log=None, level=0):
  
    if state.get('winner') is not None or depth == 0:
        score = evaluate(state, ai_player, strategy='defensive')
        if log is not None:
            log.append({'level': level, 'player': player_idx,
                        'type': 'LEAF', 'score': score, 'move': None})
        return score, None

    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    moves = valid_moves if valid_moves else [None]

    if is_maximizing:
        best_score = -math.inf
        best_move = None
        for move in moves:
            new_state = apply_move(state, player_idx, move)
            next_player = new_state['current']
            next_is_max = (next_player == ai_player)
            score, _ = minimax(new_state, depth - 1, next_player,
                               next_is_max, ai_player, log, level + 1)
            if log is not None:
                log.append({'level': level, 'player': player_idx,
                            'type': 'MAX', 'score': score, 'move': move})
            if score > best_score:
                best_score = score
                best_move = move
        return best_score, best_move
    else:
        best_score = math.inf
        best_move = None
        for move in moves:
            new_state = apply_move(state, player_idx, move)
            next_player = new_state['current']
            next_is_max = (next_player == ai_player)
            score, _ = minimax(new_state, depth - 1, next_player,
                               next_is_max, ai_player, log, level + 1)
            if log is not None:
                log.append({'level': level, 'player': player_idx,
                            'type': 'MIN', 'score': score, 'move': move})
            if score < best_score:
                best_score = score
                best_move = move
        return best_score, best_move


def minimax_decision(state, player_idx, depth=3, verbose=True):
    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)
    moves = valid_moves if valid_moves else [None]

    if verbose:
        print(f"\n[Minimax] Player {player_idx+1} considering moves (depth={depth}):")
        print(f"  Top card: {top_card}")
        print(f"  Hand: {hand}")

    best_score = -math.inf
    best_move = None

    for move in moves:
        new_state = apply_move(state, player_idx, move)
        next_player = new_state['current']
        next_is_max = (next_player == player_idx)
        score, _ = minimax(new_state, depth - 1, next_player,
                           next_is_max, player_idx)
        move_label = str(move) if move else "Draw"
        if verbose:
            print(f"    Move: {move_label:20s}  Score: {score:.1f}")
        if score > best_score:
            best_score = score
            best_move = move

    if verbose:
        print(f"  => Best move: {best_move}  (score={best_score:.1f})")
    return best_move, best_score

In [11]:
import math
random.seed(42)
state = create_initial_state()

print("  TEST 1: Basic minimax_decision call (Player 1)")

move, score = minimax_decision(state, player_idx=0, depth=3, verbose=True)
print(f"\n  Final chosen move : {move}")
print(f"  Final score       : {score:.1f}")

print("  TEST 2: Minimax when NO valid moves (forced draw)")
# Manually build a state where P1 has no matching cards
bad_hand_state = create_initial_state()
bad_hand_state['top_card'] = Card('Red', 1)
bad_hand_state['hands'][0] = [Card('Blue', 2), Card('Green', 3)]  # no match
print(f"  Top card : {bad_hand_state['top_card']}")
print(f"  P1 hand  : {bad_hand_state['hands'][0]}")
move2, score2 = minimax_decision(bad_hand_state, player_idx=0, depth=3, verbose=True)
print(f"\n  Expected : Draw (None)")
print(f"  Got      : {move2}  ✓" if move2 is None else f"  Got      : {move2}  ✗")

print("  TEST 3: Minimax with Skip card in hand")
skip_state = create_initial_state()
skip_state['top_card'] = Card('Red', 5)
skip_state['hands'][0] = [Card('Red', 9), Card('Red', 'Skip'), Card('Blue', 2)]
print(f"  Top card : {skip_state['top_card']}")
print(f"  P1 hand  : {skip_state['hands'][0]}")
move3, score3 = minimax_decision(skip_state, player_idx=0, depth=3, verbose=True)
print(f"\n  Chosen   : {move3} (score={score3:.1f})")
print(f"  (Defensive AI should prefer Skip to stall opponent)")

print("  TEST 4: Game tree log (first 15 nodes)")

log = []
random.seed(42)
state4 = create_initial_state()
minimax(state4, depth=3, player_idx=0, is_maximizing=True, ai_player=0, log=log)
symbols = {'MAX': '>', 'MIN': '<', 'LEAF': '*'}
for node in log[:15]:
    indent = '  ' * node['level']
    sym = symbols.get(node['type'], '?')
    move_str = str(node['move']) if node['move'] else 'Draw'
    print(f"  {indent}{sym} [{node['type']:4s}] P{node['player']+1} | "
          f"Move: {move_str:15s} | Score: {node['score']:.1f}")
print(f"\n  Total nodes explored: {len(log)}")

print("  TEST 5: Depth=1 sanity check")
random.seed(10)
state5 = create_initial_state()
print(f"  Top card: {state5['top_card']}")
print(f"  P1 hand : {state5['hands'][0]}")
print(f"  Valid   : {get_valid_moves(state5['hands'][0], state5['top_card'])}")
minimax_decision(state5, player_idx=0, depth=1, verbose=True)

  TEST 1: Basic minimax_decision call (Player 1)

[Minimax] Player 1 considering moves (depth=3):
  Top card: Blue 6
  Hand: [Red Skip, Blue 7, Red 8, Red 5, Blue Skip]
    Move: Blue 7                Score: 48.0
    Move: Blue Skip             Score: 47.5
  => Best move: Blue 7  (score=48.0)

  Final chosen move : Blue 7
  Final score       : 48.0
  TEST 2: Minimax when NO valid moves (forced draw)
  Top card : Red 1
  P1 hand  : [Blue 2, Green 3]

[Minimax] Player 1 considering moves (depth=3):
  Top card: Red 1
  Hand: [Blue 2, Green 3]
    Move: Draw                  Score: 43.0
  => Best move: None  (score=43.0)

  Expected : Draw (None)
  Got      : None  ✓
  TEST 3: Minimax with Skip card in hand
  Top card : Red 5
  P1 hand  : [Red 9, Red Skip, Blue 2]

[Minimax] Player 1 considering moves (depth=3):
  Top card: Red 5
  Hand: [Red 9, Red Skip, Blue 2]
    Move: Red 9                 Score: 51.0
    Move: Red Skip              Score: 46.0
  => Best move: Red 9  (score=51.0)

  C

(Blue 7, 39.0)